In [ ]:
import os
from dotenv import load_dotenv

use_databricks = False

if use_databricks:
    os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(
        scope="MainSecretScope", key="OPENAI_API_KEY"
    )
    os.environ["GOOGLE_API_KEY"] = dbutils.secrets.get(
        scope="MainSecretScope", key="GOOGLE_API_KEY"
    )
else:
    load_dotenv()

In [ ]:
from langchain.agents import create_agent

provider = "google_genai"  # "google_genai" / "openai"

if provider == "google_genai":
    model = "google_genai:gemini-3-flash-preview"
else:
    model = "gpt-5-mini"

agent = create_agent(model)

In [23]:
def moderate_ad(ad_text):

    prompt = f"""
        Esti un moderator de continut profesionist.

        Analizeaza urmatorul text al unui anunt adaugat de un mester pe o platforma de servicii si determina daca este:
        - Gibberish (text fara sens)
        - Spam (text care incearca sa promoveze ceva sau sa atraga atentia in mod nejustificat)
        - Inappropriate (text care contine limbaj ofensator, discriminare, sau alte elemente nepotrivite)
        - Language (limba in care este scris textul)

        "{ad_text}"

        Returneaza DOAR JSON valid in acest format:

        {{
        "is_gibberish": true/false,
        "is_spam": true/false,
        "is_inappropriate": true/false,
        "language": "detected language",
        "confidence": 0.0-1.0,
        "reason": "short explanation"
        }}
        """

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": prompt}
        ]
    })

    return response["messages"][-1].content

In [33]:
ads = [
    # "Mester cu experienta de 15 ani, ma ocup cu diferite lucrari de renovare, in principal zugravit si gresie / faianta.",
    # "Impreuna cu o echipa de 2 - 3 persoane, ne ocupam de reparatii de centrale si instalatii termice, precum si montarea unora noi.",
    # """Suntem o echipă cu experiență în domeniul construcțiilor, specializată în realizarea de case la roșu, la gri și la cheie, dar și în diverse lucrări de construcții civile. Punem accent pe calitatea materialelor, pe soluții adaptate fiecărui proiect și pe respectarea termenelor stabilite. Ne ocupăm de toate etapele lucrării, de la fundație până la finisajele finale, oferind profesionalism, seriozitate și atenție la detalii.

    # Executăm lucrări de zidărie și turnări de beton, precum și tencuieli și gleturi. Realizăm și montaj de acoperișuri complete, inclusiv accesorii și sisteme pluviale.

    # De asemenea, montăm garduri, realizăm pavaje și alei, iar gardurile pot fi finisate prin vopsire pentru un aspect durabil și estetic.

    # În interior, efectuăm lucrări de gletuire și zugrăveli cu vopsea lavabilă sau decorativă. Lucrăm cu structuri din rigips și tavane casetate, realizăm placări ceramice și montăm gresie și faianță. Oferim și servicii de montaj pentru parchet, blaturi și alte elemente de finisaj interior.""",
    "Cumpara acum Bitcoin, castiga 50% pe luna, oferta limitata!",
    # "asd kj h asd k jh 1231 23",
    # "Du-te-n mortii ma-tii tu si platforma ta"
]

In [34]:
for ad in ads:
    result = moderate_ad(ad)
    print("Ad:", ad)
    print("Result:", result)
    print("----")

Ad: Cumpara acum Bitcoin, castiga 50% pe luna, oferta limitata!
Result: ```json
{
  "is_gibberish": false,
  "is_spam": true,
  "is_inappropriate": false,
  "language": "Romanian",
  "confidence": 0.99,
  "reason": "The text promotes an unrelated financial product (Bitcoin investment with unrealistic returns) on a platform intended for craftsmen's services, making it unsolicited commercial content or a potential scam."
}
```
----
